# Notebook 3 — Suzlon Energy: Feature Creation and Registration

**ML use case selected from Assignment 1:** Turbine/component failure prediction.

Assignment 1 proposed SCADA sensor data for this use case. The final star schema supplied by the group does not contain raw SCADA sensor fields such as vibration or gearbox temperature, so this implementation uses the operational proxies that are actually available in the approved schema: generation, availability and service history. No model is trained, as required by the assignment.

In [0]:
CATALOG="workspace"; SCHEMA="default"
def t(name): return f"{CATALOG}.{SCHEMA}.{name}"

FEATURE_END_DATE = "2025-12-31"
LOOKBACK_DAYS = 90


## 1. Feature definitions

- `avg_availability_pct`: mean turbine availability over the last 90 days. Repeated availability loss can indicate developing reliability problems.
- `total_downtime_hours_90d`: cumulative downtime in the last 90 days. Higher downtime indicates operational instability.
- `service_ticket_count_90d`: number of service events in the last 90 days. Repeated interventions can be an early warning of failure risk.
- `avg_energy_kwh_90d`: mean daily energy generated in the last 90 days. A persistent output decline can indicate deterioration.

`turbine_id` is the primary key. `feature_date` documents the as-of date for the feature snapshot.

In [0]:
features = spark.sql(f"""
WITH generation_90d AS (
    SELECT
        g.turbine_id,
        AVG(g.availability_pct) AS avg_availability_pct,
        SUM(g.downtime_hours) AS total_downtime_hours_90d,
        AVG(g.energy_kwh) AS avg_energy_kwh_90d
    FROM {t("silver_fact_generation")} g
    JOIN {t("silver_dim_date")} d ON g.date_id=d.date_id
    WHERE d.full_date BETWEEN date_sub(DATE('{FEATURE_END_DATE}'), {LOOKBACK_DAYS}-1)
                           AND DATE('{FEATURE_END_DATE}')
    GROUP BY g.turbine_id
),
service_90d AS (
    SELECT
        s.turbine_id,
        COUNT(*) AS service_ticket_count_90d
    FROM {t("silver_fact_service")} s
    JOIN {t("silver_dim_date")} d ON s.date_id=d.date_id
    WHERE d.full_date BETWEEN date_sub(DATE('{FEATURE_END_DATE}'), {LOOKBACK_DAYS}-1)
                           AND DATE('{FEATURE_END_DATE}')
    GROUP BY s.turbine_id
)
SELECT
    t.turbine_id,
    DATE('{FEATURE_END_DATE}') AS feature_date,
    ROUND(COALESCE(g.avg_availability_pct,0),2) AS avg_availability_pct,
    ROUND(COALESCE(g.total_downtime_hours_90d,0),2) AS total_downtime_hours_90d,
    COALESCE(s.service_ticket_count_90d,0) AS service_ticket_count_90d,
    ROUND(COALESCE(g.avg_energy_kwh_90d,0),2) AS avg_energy_kwh_90d
FROM {t("silver_dim_turbine")} t
LEFT JOIN generation_90d g ON t.turbine_id=g.turbine_id
LEFT JOIN service_90d s ON t.turbine_id=s.turbine_id
""")
display(features.limit(20))


turbine_id,feature_date,avg_availability_pct,total_downtime_hours_90d,service_ticket_count_90d,avg_energy_kwh_90d
16,2025-12-31,96.11,84.13,3,35280.88
39,2025-12-31,97.24,59.51,4,15562.53
40,2025-12-31,97.81,47.32,12,43000.56
45,2025-12-31,95.67,93.43,5,14443.11
46,2025-12-31,97.16,61.25,5,33215.60
74,2025-12-31,96.75,70.27,7,13150.37
122,2025-12-31,96.20,82.01,8,21088.66
125,2025-12-31,94.80,112.40,7,38066.86
130,2025-12-31,97.55,52.88,8,33972.56
142,2025-12-31,96.57,74.13,9,40226.30


## 2. Data-quality checks

The feature table must contain one row per turbine, a non-null primary key and non-negative operational measures.

In [0]:
assert features.select("turbine_id").distinct().count() == features.count()
assert features.filter("turbine_id IS NULL").count() == 0
assert features.filter("avg_availability_pct < 0 OR avg_availability_pct > 100 OR total_downtime_hours_90d < 0 OR service_ticket_count_90d < 0 OR avg_energy_kwh_90d < 0").count() == 0
print("Feature checks passed:", features.count(), "turbines")


Feature checks passed: 1000 turbines


## 3. Register the feature table

Unity Catalog feature tables are Delta tables with a primary key. The current Feature Engineering client is used below. If it is not pre-installed on the selected runtime, install `databricks-feature-engineering` first.

In [0]:
%pip install databricks-feature-engineering==0.17.1

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

feature_table_name = t("feature_turbine_failure")

fe = FeatureEngineeringClient()

try:
    fe.create_table(
        name=feature_table_name,
        primary_keys=["turbine_id"],
        df=features,
        description="Suzlon 90-day turbine operational features for future failure-risk prediction."
    )
except Exception as e:
    print("Create table message:", e)

fe.write_table(
    name=feature_table_name,
    df=features,
    mode="merge"
)

display(fe.read_table(name=feature_table_name).limit(20))


2026/09/15 13:35:21 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['turbine_id'] of table 'workspace.default.feature_turbine_failure' to NOT NULL.
2026/09/15 13:35:23 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['turbine_id'] on table 'workspace.default.feature_turbine_failure'.
2026/09/15 13:35:31 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'workspace.default.feature_turbine_failure'.


turbine_id,feature_date,avg_availability_pct,total_downtime_hours_90d,service_ticket_count_90d,avg_energy_kwh_90d
16,2025-12-31,96.11,84.13,3,35280.88
39,2025-12-31,97.24,59.51,4,15562.53
40,2025-12-31,97.81,47.32,12,43000.56
45,2025-12-31,95.67,93.43,5,14443.11
46,2025-12-31,97.16,61.25,5,33215.60
74,2025-12-31,96.75,70.27,7,13150.37
122,2025-12-31,96.20,82.01,8,21088.66
125,2025-12-31,94.80,112.40,7,38066.86
130,2025-12-31,97.55,52.88,8,33972.56
142,2025-12-31,96.57,74.13,9,40226.30


## 4. Business interpretation

These features convert historical operational behaviour into an ML-ready turbine-level representation. In a future failure model, low availability, high downtime, repeated service tickets and declining energy output could provide signals of elevated failure risk. The assignment only requires the feature table, so no model training is performed.